# 06 · Find unusual reviews (anomaly detection, unsupervised)

**Use case:** the marketplace team wants a first pass at suspicious reviews — text that does not fit the product, ratings out of line with the words — without labelled examples.

**Model / lane:** GraphMAE (unsupervised lane), anomaly scoring on the node embeddings

**Sub-tasks**
1. Define an unsupervised **anomaly detection** task on `review`
2. Train GraphMAE and score reviews by reconstruction error
3. Read what the run reports (`val_loss`, epochs)
4. Score a sample of reviews: which are flagged, with what score

In [1]:
import sys, json
sys.path.insert(0, "..")
from _common import connect, get_or_create_project, print_schema, train_or_reuse, credits_used, save_metrics, show

ls = connect()

signed in as team@langsat.ai · tier team · key 'TEST_SDK_2' with 17 scopes


In [2]:
p = get_or_create_project(ls, "review-anomalies", kind="data_science")
before = credits_used(ls)

reusing project 828e4626-323e-4bee-9de4-7c825f7835d3 (amazon-reviews-review-anomalies, status=ready)
project ready: status=ready · type=data_science · files=['customer.csv', 'product.csv', 'review.csv']


In [3]:
model = train_or_reuse(p, "Detect unusual or suspicious reviews based on their text, rating and the product and customer they belong to",
                       task_type="unsupervised", subtask_type="anomaly_detection", enable_text_embedding=True)
metrics = model["metrics"]
show(metrics)

reusing model 1489addf-1a1b-46c5-874d-afcbd9297d66 (unsupervised, trained 2026-09-15T09:02:48.659250+00:00)
  epoch                  100
  val_loss               0.6214


Anomaly detection is self-supervised too: a review whose embedding the model reconstructs badly is unusual for its neighbourhood (its product, its customer, its words). Score a sample (50 credits each, 20 here) and look at the flagged ones.

In [4]:
import pandas as pd
review = p.table("review").to_pandas()
sample = review.sample(20, random_state=7)
ids = sample.index.tolist()                      # review's key is the synthetic row id
batch = ls.predict.predict_batch(model_id=model["model_id"], entity_ids=ids)
sample = sample.assign(is_anomaly=[pr.get("is_anomaly") for pr in batch["predictions"]],
                       score=[pr.get("anomaly_score") for pr in batch["predictions"]])
print("flagged:", int(sample["is_anomaly"].fillna(False).astype(bool).sum()), "of", len(sample), "· threshold:", (batch["predictions"][0] or {}).get("threshold"))
for _, r in sample.sort_values("score", ascending=False).head(5).iterrows():
    print(f"  score {r['score']:.3f} · {r['rating']}★ · {str(r['summary'])[:40]!r} · {str(r['review_text'])[:90]!r}")

flagged: 1 of 20 · threshold: 0.4351722179223622
  score 0.721 · 1★ · 'The author might do better attending her' · "Stars? None! I'm still agonizing to finish this verbosity monster. The author might do bet"
  score 0.288 · 4★ · "The end of the author's road" · 'This book gave a terrific description of life in Scotland, probably, back in the early 19t'
  score 0.258 · 4★ · 'Four Stars' · 'Good book. Wrapped up the story nicely'
  score 0.198 · 4★ · 'L' · 'I spent my teenage years in Minneapolis, right near the setting for this book.  I remember'
  score 0.161 · 5★ · 'Wonderful read!' · 'I loved this book!  It was a nice, quick, easy read.  I had a hard time putting it down.  '


In [5]:
charged = credits_used(ls) - before
save_metrics(".", {"notebook": "06_review_anomaly", "task": "anomaly detection · review (unsupervised)", "model": model.get("model_type"),
                   "project_id": p.id, "model_id": model["model_id"], "training_duration_sec": model.get("training_duration_sec"),
                   "credits_charged_this_run": charged,
                   "headline": {"val_loss": metrics.get("val_loss"), "epochs": metrics.get("epoch"), "flagged_in_sample": int(sample["is_anomaly"].fillna(False).astype(bool).sum()), "sample_size": len(sample), "note": "self-supervised: reconstruction loss, no labels"}})

wrote results/metrics.json


PosixPath('results/metrics.json')